# AI 2002 - Assignment #2: GAME AI
## Simplified UNO Game with Adversarial Search

**Faculty:** Ms. Umarah Qaseem  
**Assigned:** 18 March 2026  
**Deadline:** 29 March 2026  

**Student:** Hafsa Tanveer 

**ID:** 24i-2452

In my this notebook I implements a simplified UNO game with 3 players using Minimax and Expectimax search algorithms.

### Game Setup
- **Player 1:** AI using Minimax (Defensive Strategy)
- **Player 2:** AI using Expectimax (Offensive Strategy)  
- **Player 3:** Manual Mode or Simulation Mode (using Minimax)

Each player starts with 5 cards. Deck contains Red/Blue/Green/Yellow 0-9 and Skip cards.

### Rules
1. Play if same color or same number
2. Draw 1 card if no valid move
3. Skip skips next player
4. Win when hand is empty

## 1. Card Class Implementation

We define a Card class to represent UNO cards with color and value 

In [1]:
import random
import copy
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict, Any

# -----------------------------
# Card and utility definitions
# -----------------------------
COLORS = ["Red", "Blue", "Green", "Yellow"]
SKIP_LABEL = "Skip"

@dataclass(frozen=True)
class Card:
    color: str
    value: Any  # int 0-9 OR "Skip"

    def __str__(self):
        return f"{self.color} {self.value}"

## 2. Deck Generator

Function to create and shuffle the deck with 4 colors, 0-9 numbers, and 2 Skip cards per color.

In [2]:
def create_deck(skip_per_color: int = 2) -> List[Card]:
    """
    Create simplified UNO deck:
    - 4 colors, each has number cards 0..9 (one each)
    - Skip cards per color (default 2 each)
    """
    deck = []
    for c in COLORS:
        for n in range(10):
            deck.append(Card(c, n))
        for _ in range(skip_per_color):
            deck.append(Card(c, SKIP_LABEL))
    random.shuffle(deck)
    return deck

## 3. Legal Move Generator

Function to get valid moves based on same color or same number/value.

In [3]:
def is_valid_move(card: Card, top_card: Card) -> bool:
    """
    Valid if same color OR same number/value
    """
    return (card.color == top_card.color) or (card.value == top_card.value)

def get_valid_moves(hand: List[Card], top_card: Card) -> List[Card]:
    return [c for c in hand if is_valid_move(c, top_card)]

## 4. Game State Representation

State is a dictionary with hands, top_card, deck, current_player, skip_next.

In [4]:
def build_state(hands: List[List[Card]], top_card: Card, deck: List[Card], current_player: int, skip_next: bool = False) -> Dict:
    """
    current_player in {0,1,2}
    hands[0] = P1
    hands[1] = P2
    hands[2] = P3
    """
    return {
        "hands": [list(h) for h in hands],  # deep-ish copy
        "top_card": top_card,
        "deck": list(deck),
        "current_player": current_player,
        "skip_next": skip_next
    }

def state_deepcopy(state: Dict) -> Dict:
    return {
        "hands": [list(h) for h in state["hands"]],
        "top_card": state["top_card"],
        "deck": list(state["deck"]),
        "current_player": state["current_player"],
        "skip_next": state["skip_next"]
    }

def next_player_idx(i: int) -> int:
    return (i + 1) % 3

def terminal_winner(state: Dict) -> Optional[int]:
    for i in range(3):
        if len(state["hands"][i]) == 0:
            return i
    return None

## 5. State Transition Function

apply_move updates the state after a move (Card or None for draw).

In [5]:
def apply_move(state: Dict, move: Optional[Card]) -> Dict:
    """
    move is Card => play that card
    move is None => draw one
    Applies action for current_player, then advances turn.
    Handles skip behavior.
    """
    st = state_deepcopy(state)
    p = st["current_player"]
    hand = st["hands"][p]

    # If current turn is skipped it is set by previous skip card
    if st["skip_next"]:
        st["skip_next"] = False
        st["current_player"] = next_player_idx(p)
        return st

    if move is None:
        # Draw one card if available
        if st["deck"]:
            hand.append(st["deck"].pop())
    else:
        # Play chosen card
        hand.remove(move)
        st["top_card"] = move
        if move.value == SKIP_LABEL:
            st["skip_next"] = True

    # Advance turn
    st["current_player"] = next_player_idx(p)
    return st

## 6. Evaluation Function

Score = 50 − 5(CAI) + 2(Copp) + 3(S)

Tuned for Defensive (P1) and Offensive (P2) strategies.

In [6]:
def baseline_score(state: Dict, ai_idx: int) -> float:
    """
    Score = 50 − 5(CAI) + 2(Copp) + 3(S)
    Copp = average cards of two opponents
    """
    my_cards = len(state["hands"][ai_idx])
    opps = [i for i in range(3) if i != ai_idx]
    opp_avg = (len(state["hands"][opps[0]]) + len(state["hands"][opps[1]])) / 2.0
    skips = sum(1 for c in state["hands"][ai_idx] if c.value == SKIP_LABEL)
    return 50 - 5 * my_cards + 2 * opp_avg + 3 * skips

def evaluate_defensive(state: Dict, ai_idx: int) -> float:
    """
    Defensive tuning:
    - Keep own cards low, but stronger penalty if opponent near win
    - Keep skip cards valuable
    """
    base = baseline_score(state, ai_idx)

    opps = [i for i in range(3) if i != ai_idx]
    opp_sizes = [len(state["hands"][o]) for o in opps]
    min_opp = min(opp_sizes)

    # Extra defensive terms
    threat_penalty = 0
    if min_opp <= 2:
        threat_penalty = (3 - min_opp) * 12  # strong penalty when someone is near winning

    my_skips = sum(1 for c in state["hands"][ai_idx] if c.value == SKIP_LABEL)
    skip_bonus = 2.5 * my_skips

    return base + skip_bonus - threat_penalty

def evaluate_offensive(state: Dict, ai_idx: int) -> float:
    """
    Offensive tuning:
    - Strongly reward reducing own cards quickly
    - Less concern for holding skip
    """
    my_cards = len(state["hands"][ai_idx])
    opps = [i for i in range(3) if i != ai_idx]
    opp_avg = (len(state["hands"][opps[0]]) + len(state["hands"][opps[1]])) / 2.0
    skips = sum(1 for c in state["hands"][ai_idx] if c.value == SKIP_LABEL)

    # Tuned offensive weighting
    return 55 - 7 * my_cards + 1.5 * opp_avg + 1.0 * skips

## 7. Minimax Search Algorithm (Defensive for P1)

Depth 3, opponents minimize AI score.

In [7]:
def minimax_value(state: Dict, depth: int, root_ai_idx: int) -> float:
    winner = terminal_winner(state)
    if winner is not None:
        if winner == root_ai_idx:
            return 1000.0
        else:
            return -1000.0
    if depth == 0:
        return evaluate_defensive(state, root_ai_idx)

    current = state["current_player"]

    # If this player's turn is skipped, just apply skip pass
    if state["skip_next"]:
        child = apply_move(state, None)  
        return minimax_value(child, depth - 1, root_ai_idx)

    hand = state["hands"][current]
    valid = get_valid_moves(hand, state["top_card"])

    # If no valid moves than draw forced
    actions = valid if valid else [None]

    # Root AI turn => maximize
    if current == root_ai_idx:
        best = float("-inf")
        for a in actions:
            child = apply_move(state, a)
            val = minimax_value(child, depth - 1, root_ai_idx)
            best = max(best, val)
        return best
    else:
        # Opponents minimize root AI score 
        best = float("inf")
        for a in actions:
            child = apply_move(state, a)
            val = minimax_value(child, depth - 1, root_ai_idx)
            best = min(best, val)
        return best

def minimax_decision(state: Dict, ai_idx: int, depth: int = 3, verbose: bool = True) -> Optional[Card]:
    hand = state["hands"][ai_idx]
    valid = get_valid_moves(hand, state["top_card"])
    actions = valid if valid else [None]

    scored = []
    for a in actions:
        child = apply_move(state, a)
        v = minimax_value(child, depth - 1, ai_idx)
        scored.append((a, v))

    scored.sort(key=lambda x: x[1], reverse=True)
    best_move = scored[0][0]

    if verbose:
        print(f"\n[P{ai_idx+1} Minimax Defensive] All possible decisions considered at depth {depth}:")
        for mv, sc in scored:
            move_str = "Draw" if mv is None else f"Play: {mv}"
            print(f"{move_str:20s} -> Score: {sc:.2f}")

    return best_move

## 8. Expectimax Search Algorithm (Offensive for P2)

Depth 3, draw as chance node.

In [8]:
def expectimax_value(state: Dict, depth: int, root_ai_idx: int) -> float:
    winner = terminal_winner(state)
    if winner is not None:
        if winner == root_ai_idx:
            return 1000.0
        else:
            return -1000.0
    if depth == 0:
        return evaluate_offensive(state, root_ai_idx)

    current = state["current_player"]

    # Skip handling
    if state["skip_next"]:
        child = apply_move(state, None)
        return expectimax_value(child, depth - 1, root_ai_idx)

    hand = state["hands"][current]
    valid = get_valid_moves(hand, state["top_card"])

    # Root AI turn, MAX among legal plays or chance draw if no legal
    if current == root_ai_idx:
        if valid:
            return max(expectimax_value(apply_move(state, mv), depth - 1, root_ai_idx) for mv in valid)
        else:
            # chancer node: it draw random card from deck
            deck = state["deck"]
            if not deck: # no card to draw, pass
                child = apply_move(state, None)
                return expectimax_value(child, depth - 1, root_ai_idx)

            total = 0.0
            prob = 1.0 / len(deck)
            for i, card in enumerate(deck):
                # simulate drawing specific card uniformly
                st = state_deepcopy(state)
                st["hands"][current].append(card)
                st["deck"].pop(i)
                # After draw, turn ends in this simplified game
                st["current_player"] = next_player_idx(current)
                total += prob * expectimax_value(st, depth - 1, root_ai_idx)
            return total

    # Opponent nodes P1/P3 for P2's tree
    else:
        if valid:
            prob = 1.0 / len(valid)
            total = 0.0
            for mv in valid:
                child = apply_move(state, mv)
                total += prob * expectimax_value(child, depth - 1, root_ai_idx)
            return total
        else:
            # no legal than forced draw 
            deck = state["deck"]
            if not deck:
                child = apply_move(state, None)
                return expectimax_value(child, depth - 1, root_ai_idx)

            total = 0.0
            prob = 1.0 / len(deck)
            for i, card in enumerate(deck):
                st = state_deepcopy(state)
                st["hands"][current].append(card)
                st["deck"].pop(i)
                st["current_player"] = next_player_idx(current)
                total += prob * expectimax_value(st, depth - 1, root_ai_idx)
            return total

def expectimax_decision(state: Dict, ai_idx: int, depth: int = 3, verbose: bool = True) -> Optional[Card]:
    hand = state["hands"][ai_idx]
    valid = get_valid_moves(hand, state["top_card"])

    if valid:
        scored = []
        for mv in valid:
            child = apply_move(state, mv)
            v = expectimax_value(child, depth - 1, ai_idx)
            scored.append((mv, v))
        scored.sort(key=lambda x: x[1], reverse=True)
        best_move = scored[0][0]

        if verbose:
            print(f"\n[P{ai_idx+1} Expectimax Offensive] All possible decisions considered at depth {depth}:")
            for mv, sc in scored:
                print(f"Play: {mv:15s} -> Expected score: {sc:.2f}" if isinstance(mv, str) else f"Play: {mv} -> Expected score: {sc:.2f}")
        return best_move

    # No valid move must draw
    if verbose:
        print(f"\n[P{ai_idx+1} Expectimax Offensive] No valid move -> Draw")
    return None

## 9. Player 3 Modes

Manual input or simulation using Minimax.

In [9]:
def manual_choose_move(hand: List[Card], top_card: Card) -> Optional[Card]:
    valid = get_valid_moves(hand, top_card)
    print("\nYour hand:")
    for i, c in enumerate(hand):
        mark = " (valid)" if c in valid else ""
        print(f"{i}: {c}{mark}")

    if not valid:
        print("No valid card. You must draw.")
        return None

    while True:
        choice = input("Enter card index to play OR 'd' to draw: ").strip().lower()
        if choice == 'd':
            return None
        if choice.isdigit():
            idx = int(choice)
            if 0 <= idx < len(hand):
                if hand[idx] in valid:
                    return hand[idx]
                else:
                    print("Invalid move by rules. Pick a valid card.")
            else:
                print("Index out of range.")
        else:
            print("Invalid input.")

def print_hand(player_name: str, hand: List[Card]):
    cards = ", ".join(str(c) for c in hand)
    print(f"{player_name} hand ({len(hand)}): {cards}")

def print_state(state: Dict):
    print("\n" + "=" * 60)
    print(f"Top card: {state['top_card']}")
    print(f"Current turn: P{state['current_player']+1}")
    print(f"Deck remaining: {len(state['deck'])}")
    for i in range(3):
        print_hand(f"P{i+1}", state["hands"][i])
    if state["skip_next"]:
        print("NOTE: Next player turn will be skipped.")

## 10. Game Simulation Loop

Runs the full game with chosen mode for P3.

In [10]:
def play_game(p3_mode: str = "simulation", seed: int = 42, max_turns: int = 200, depth: int = 3):
    random.seed(seed)

    deck = create_deck(skip_per_color=2)

    # Deal 5 cards to each player
    hands = [[], [], []]
    for _ in range(5):
        for p in range(3):
            hands[p].append(deck.pop())

    # Choose initial top card, avoid skip
    top = deck.pop()
    while top.value == SKIP_LABEL and deck:
        deck.insert(0, top)
        random.shuffle(deck)
        top = deck.pop()

    state = build_state(hands, top, deck, current_player=0, skip_next=False)

    print("\n===== GAME START =====")
    print_state(state)

    turn = 0
    while turn < max_turns:
        winner = terminal_winner(state)
        if winner is not None:
            print(f"\n Winner is Player {winner+1}")
            break

        p = state["current_player"]

        # If turn will be skipped
        if state["skip_next"]:
            print(f"\nPlayer {p+1} turn is skipped due to Skip card.")
            state = apply_move(state, None)
            turn += 1
            continue

        print(f"\n--- Turn {turn+1} | Player {p+1} ---")
        print(f"Top card: {state['top_card']}")

        if p == 0:
            move = minimax_decision(state, ai_idx=0, depth=depth, verbose=True)
        elif p == 1:
            move = expectimax_decision(state, ai_idx=1, depth=depth, verbose=True)
        else:
            if p3_mode == "manual":
                move = manual_choose_move(state["hands"][2], state["top_card"])
            else:
                move = minimax_decision(state, ai_idx=2, depth=depth, verbose=True)

        before_top = state["top_card"]
        state = apply_move(state, move)

        if move is None:
            print(f"Player {p+1} action: Draw 1 card")
        else:
            print(f"Player {p+1} action: Play {move}")
            if move.value == SKIP_LABEL:
                print("Skip effect activated: next player's turn will be skipped.")

        print(f"Top card was {before_top}, now {state['top_card']}")
        print(f"Cards count -> P1:{len(state['hands'][0])} P2:{len(state['hands'][1])} P3:{len(state['hands'][2])}")

        turn += 1

    if turn >= max_turns:
        print("\nGame reached max turns. Draw/stop condition for demo.")

    print("\n===== GAME END =====")
    print_state(state)
    return state

## 11. Game Tree Generation

Text-based tree for submission evidence.

In [11]:
def generate_tree_text(state: Dict, root_player: int, algorithm: str = "minimax", depth: int = 2, indent: str = "") -> List[str]:
    """
    Generates a text representation of game tree (for submission evidence).
    algorithm: 'minimax' or 'expectimax'
    """
    lines = []
    winner = terminal_winner(state)
    if winner is not None:
        lines.append(f"{indent}Terminal -> Winner P{winner+1}")
        return lines

    if depth == 0:
        if algorithm == "minimax":
            val = evaluate_defensive(state, root_player)
        else:
            val = evaluate_offensive(state, root_player)
        lines.append(f"{indent}Leaf Eval = {val:.2f}")
        return lines

    p = state["current_player"]
    lines.append(f"{indent}Node: P{p+1} turn | Top: {state['top_card']} | Deck:{len(state['deck'])}")

    if state["skip_next"]:
        lines.append(f"{indent}  [Skip turn applied]")
        child = apply_move(state, None)
        lines.extend(generate_tree_text(child, root_player, algorithm, depth - 1, indent + "    "))
        return lines

    valid = get_valid_moves(state["hands"][p], state["top_card"])
    actions = valid if valid else [None]

    for a in actions:
        label = "Draw" if a is None else f"Play {a}"
        lines.append(f"{indent}  Action -> {label}")
        child = apply_move(state, a)
        lines.extend(generate_tree_text(child, root_player, algorithm, depth - 1, indent + "    "))

    return lines

## Evaluation Function Design

In this assignment, two different AI strategies are used for the UNO game:

- **Player 1**: Minimax with a **Defensive** strategy  
- **Player 2**: Expectimax with an **Offensive** strategy  

The baseline evaluation formula given in the assignment is:

\[
\text{Score} = 50 - 5(CAI) + 2(Copp) + 3(S)
\]

Where:

- \(CAI\): Number of cards in current AI player's hand  
- \(Copp\): Average number of cards held by the two opponents  
- \(S\): Number of Skip cards in AI hand  



### Baseline Intuition

- The term \(-5(CAI)\) encourages reducing own hand size.
- The term \(+2(Copp)\) rewards states where opponents still have many cards.
- The term \(+3(S)\) rewards keeping special Skip cards.

So higher score = better state for the current AI.



### Defensive Evaluation (Used by Player 1 - Minimax)

Defensive AI focuses on:
1. Preventing opponents from reaching 0 quickly.
2. Preserving Skip cards for control.
3. Blocking dangerous states where any opponent is near winning.

Therefore, in addition to baseline scoring, an extra **threat penalty** is applied if an opponent has very few cards (especially 1 or 2).  
A small extra bonus is also added for Skip card retention.

This makes Player 1 prefer safer moves that reduce opponent momentum instead of only dumping cards aggressively.



### Offensive Evaluation (Used by Player 2 - Expectimax)

Offensive AI focuses on:
1. Fast card shedding (winning quickly).
2. High-tempo play with less long-term caution.
3. Maximizing expected gain under uncertainty (draw outcomes).

For this strategy:
- Penalty for own cards is made stronger than baseline.
- Opponent-card influence is reduced.
- Skip card bonus is smaller than defensive mode.

This causes Player 2 to choose moves that reduce its hand count quickly, even if they are less conservative.



### Why Two Different Evaluations?

If both players use exactly the same evaluation, both AIs behave similarly.  
The assignment requires distinct logic for **Defensive (Minimax)** and **Offensive (Expectimax)** behavior.  
Hence, weight tuning is necessary to create clearly different play styles and decision patterns.

## Game Simulation Output

Running a sample game in simulation mode.

In [12]:
final_state = play_game(p3_mode="simulation", seed=7, max_turns=120, depth=3)


===== GAME START =====

Top card: Green 3
Current turn: P1
Deck remaining: 32
P1 hand (5): Blue 8, Yellow 5, Green Skip, Yellow 1, Blue 1
P2 hand (5): Red 9, Red 3, Red 6, Yellow 7, Red 2
P3 hand (5): Green 1, Red 4, Blue Skip, Green 8, Red 5

--- Turn 1 | Player 1 ---
Top card: Green 3

[P1 Minimax Defensive] All possible decisions considered at depth 3:
Play: Green Skip     -> Score: 39.00
Player 1 action: Play Green Skip
Skip effect activated: next player's turn will be skipped.
Top card was Green 3, now Green Skip
Cards count -> P1:4 P2:5 P3:5

Player 2 turn is skipped due to Skip card.

--- Turn 3 | Player 3 ---
Top card: Green Skip

[P3 Minimax Defensive] All possible decisions considered at depth 3:
Play: Green 8        -> Score: 44.50
Play: Green 1        -> Score: 42.50
Play: Blue Skip      -> Score: 40.00
Player 3 action: Play Green 8
Top card was Green Skip, now Green 8
Cards count -> P1:4 P2:5 P3:4

--- Turn 4 | Player 1 ---
Top card: Green 8

[P1 Minimax Defensive] All po

## Sample Game Tree

Generated from an initial state for P1 Minimax.

In [13]:
random.seed(42)
deck = create_deck(skip_per_color=2)
hands = [[], [], []]
for _ in range(5):
    for p in range(3):
        hands[p].append(deck.pop())
top = deck.pop()
initial_state = build_state(hands, top, deck, current_player=0, skip_next=False)

print("\n\n===== SAMPLE TREE (P1 Minimax, depth=2) =====")
tree_lines_1 = generate_tree_text(initial_state, root_player=0, algorithm="minimax", depth=2)
for ln in tree_lines_1[:50]:  # limit output
    print(ln)

print("\n\n===== SAMPLE TREE (P2 Expectimax, depth=2) =====")
tree_lines_2 = generate_tree_text(initial_state, root_player=1, algorithm="expectimax", depth=2)
for ln in tree_lines_2[:50]:
    print(ln)



===== SAMPLE TREE (P1 Minimax, depth=2) =====
Node: P1 turn | Top: Blue 1 | Deck:32
  Action -> Play Blue 5
    Node: P2 turn | Top: Blue 5 | Deck:32
      Action -> Play Blue 3
        Leaf Eval = 39.00


===== SAMPLE TREE (P2 Expectimax, depth=2) =====
Node: P1 turn | Top: Blue 1 | Deck:32
  Action -> Play Blue 5
    Node: P2 turn | Top: Blue 5 | Deck:32
      Action -> Play Blue 3
        Leaf Eval = 33.75


## Comparison of Minimax (Defensive) vs Expectimax (Offensive)

### Strategy Employed

- **Player 1 (Minimax - Defensive):**
  - Assumes opponents act in ways that are worst for Player 1.
  - Prioritizes risk control and opponent blocking.
  - Tries to preserve Skip cards for tactical interruption.

- **Player 2 (Expectimax - Offensive):**
  - Models uncertainty through expected values.
  - Treats draw scenarios as chance events.
  - Prioritizes aggressive card reduction and fast completion.



### Observed Behavior in Simulation

1. **Minimax Defensive behavior**
   - More cautious decisions.
   - Often avoids moves that may allow quick opponent recovery.
   - Performs well when opponents are close to winning because threat control is explicit.

2. **Expectimax Offensive behavior**
   - More aggressive and tempo-driven.
   - Often picks actions with highest expected value under draw uncertainty.
   - Can finish quickly in favorable card distributions.



### Which Algorithm Performed Better?

In my simulation runs, performance depends on game state:

- **Expectimax** performed better in open, uncertain situations with many deck possibilities, because expected-value reasoning helped it choose stronger average outcomes.
- **Minimax** performed better in high-risk late-game states (when an opponent had very few cards), because its defensive assumptions and threat penalties prevented sudden losses.



### Why the Better Performance Occurred

- Expectimax has an advantage when uncertainty dominates (draw outcomes are important).
- Minimax has an advantage when worst-case opponent pressure dominates (end-game defense).
- Therefore, there is no universally best method for all states; each is superior under different strategic conditions.



### Final Conclusion

The assignment objective of creating two distinct AI styles was achieved:

- **Minimax** produced a defensive control-oriented agent.
- **Expectimax** produced an offensive expectation-driven agent.

This comparison demonstrates that algorithm choice should match intended strategy:
- choose **Minimax** for safety and adversarial robustness,
- choose **Expectimax** for probabilistic aggression and faster card shedding.

## GitHub Repository

https://github.com/hafsaatanveer/ai.git